# SI4006 · Entrega M1 — Fine-tuning con LoRA
## Predicción del ritmo de generación de residuos en cocinas de restaurante

**Curso:** Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Módulo 1:** Arquitectura transformer y fine-tuning eficiente

---

### ¿Qué hace este sistema, en una frase?

> Mira el estado de un contenedor de basura de la cocina de un restaurante —cuán lleno está, cómo
> viene llenándose, qué hora y qué día es— y **anticipa cuánta basura va a acumular en la próxima
> hora**, clasificándola como poca (`LOW`), media (`MEDIUM`) o mucha (`HIGH`), para que el personal
> sepa cuándo va a tocar vaciarlo antes de que se desborde.

| | |
|---|---|
| **Dataset** | FIKWaste — 3 cocinas de restaurante reales en Portugal (sensores ultrasónicos, 2019) |
| **Familia** | Encoder-only |
| **Modelo base** | `distilbert-base-uncased` |
| **Tarea** | Clasificación multiclase (`LOW` / `MEDIUM` / `HIGH`) |
| **Métrica principal** | F1-macro |
| **Método de ajuste** | LoRA (PEFT) |
| **Semilla** | 42 |

---

### ⚠️ Nota metodológica honesta (léela antes de correr nada)

FIKWaste es un dataset de **sensores**, no de texto: son series temporales de distancia (cm) y
volumen (%) medidas con sensores ultrasónicos sobre las tapas de los contenedores. **No hay texto
natural en el dominio.**

Para poder usar un transformer encoder —que es lo que pide el módulo— **serializamos cada ventana
horaria a una descripción estructurada en inglés** (sección 6). Esta transformación *no crea
información nueva*: reexpresa las mismas variables en un formato que DistilBERT puede consumir.

Somos explícitos sobre esto porque es una decisión discutible y la asignación premia la honestidad
por encima del número: **para datos tabulares/de sensor, un modelo de árboles suele ganarle a un
transformer.** En la sección 17 lo verificamos empíricamente contra un `HistGradientBoosting` en vez
de esconderlo. Lo que este notebook demuestra es el *dominio de la técnica* (LoRA, Trainer API,
comparación contra baseline), no que un transformer sea la herramienta óptima para este dato.

## 1 · Setup

Requiere **GPU T4** en Colab: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.

Siguiendo el lab de S04, **no fijamos versiones** de `torch` ni `transformers` (usamos las de
Colab) e instalamos solo lo que falta. Desinstalamos `torchao` porque la versión que trae Colab
choca con `peft` al llamar `get_peft_model` y aquí no lo usamos.

In [ ]:
%pip install -q -U peft datasets accelerate scikit-learn
%pip uninstall -y -q torchao
print("Instalación lista.")

In [ ]:
import os
import glob
import io
import random
import zipfile
import warnings

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

import transformers, peft
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("torch:", torch.__version__, "| device:", device)
if device == "cpu":
    print("\n⚠️  Estás en CPU: el notebook corre, pero lento. Activa la GPU T4.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

## 2 · Obtención de los datos crudos

El dataset se descarga **directamente desde OSF** para que el notebook sea reproducible sin subir
archivos a mano. Si ya tienes la carpeta local (por ejemplo al clonar el repo), la usa y se salta la
descarga.

**Fuente:** https://osf.io/tyaj6/ · **Licencia:** CC BY 4.0
**Cita:** Pereira, L.; Aguiar, V.; Vasconcelos, F. *FIKWaste: A Waste Generation Dataset from Three
Restaurant Kitchens in Portugal.* Data 2021, 6, 25. https://doi.org/10.3390/data6030025

In [ ]:
RAW_DIR = "data/raw/FIKWaste"          # ruta dentro del repo
OSF_ZIP = "https://files.osf.io/v1/resources/tyaj6/providers/osfstorage/?zip="

def ensure_raw_data(raw_dir=RAW_DIR):
    '''Devuelve la carpeta con los datos crudos; la descarga de OSF si no existe.'''
    if os.path.isdir(raw_dir) and glob.glob(os.path.join(raw_dir, "Kitchen *", "*", "measurements.csv")):
        print(f"Usando datos locales en '{raw_dir}'.")
        return raw_dir

    print("Descargando FIKWaste desde OSF…")
    import requests
    r = requests.get(OSF_ZIP, timeout=180)
    r.raise_for_status()
    os.makedirs(raw_dir, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(raw_dir)
    print(f"Descargado y extraído en '{raw_dir}'.")
    return raw_dir

RAW_DIR = ensure_raw_data()

for path in sorted(glob.glob(os.path.join(RAW_DIR, "**", "*.csv"), recursive=True)):
    print(" ", os.path.relpath(path, RAW_DIR))

In [ ]:
def load_raw(raw_dir):
    '''Carga measurements, labels y deployments de las 3 cocinas.'''
    meas, labs = [], []

    for path in sorted(glob.glob(os.path.join(raw_dir, "Kitchen *", "*", "measurements.csv"))):
        parts = path.replace("\\", "/").split("/")
        d = pd.read_csv(path, parse_dates=["timestamp"])
        d["kitchen"], d["bin"] = parts[-3], parts[-2]
        meas.append(d)

    for path in sorted(glob.glob(os.path.join(raw_dir, "Kitchen *", "*", "labels.csv"))):
        parts = path.replace("\\", "/").split("/")
        # los labels traen BOM UTF-8
        d = pd.read_csv(path, encoding="utf-8-sig", parse_dates=["timestamp"])
        d["kitchen"], d["bin"] = parts[-3], parts[-2]
        labs.append(d)

    dep = pd.read_csv(os.path.join(raw_dir, "deployments.csv"),
                      dayfirst=True, parse_dates=["start", "end"])
    return (pd.concat(meas, ignore_index=True),
            pd.concat(labs, ignore_index=True),
            dep)

measurements, labels, deployments = load_raw(RAW_DIR)

print("measurements:", measurements.shape)
print("labels      :", labels.shape)
print("deployments :", deployments.shape)
display(measurements.head())
display(deployments)

## 3 · Exploración del dataset crudo

Antes de transformar nada, miramos qué hay. Tres cosas importan para el diseño de la tarea:

1. **Cobertura desigual:** la Cocina 1 muestrea cada **1 minuto**; las Cocinas 2 y 3 cada **5
   minutos** (para ahorrar batería). Los conteos crudos no son comparables entre cocinas.
2. **La Cocina 1 no tiene contenedor de vidrio** → son 11 series cocina×contenedor, no 12.
3. **Los labels de vaciado están incompletos a propósito:** solo se conservaron los eventos que al
   menos 2 de los 3 autores confirmaron. Hay vaciados reales sin etiquetar.

In [ ]:
resumen = (measurements.groupby(["kitchen", "bin"])
           .agg(mediciones=("volume", "size"),
                inicio=("timestamp", "min"),
                fin=("timestamp", "max"),
                vol_medio=("volume", "mean"))
           .round(1))
resumen["labels_vaciado"] = labels.groupby(["kitchen", "bin"]).size()
display(resumen)

print("Total mediciones:", len(measurements), "| Total labels:", len(labels))
print("Valores faltantes en measurements:", measurements.isna().sum().sum())
print("Rango de volume (%):", measurements.volume.min(), "-", measurements.volume.max())
print("\nFrecuencia de muestreo mediana por cocina (segundos):")
for k, g in measurements.groupby("kitchen"):
    dt = g.sort_values("timestamp").groupby("bin").timestamp.diff().dt.total_seconds()
    print(f"  {k}: {dt.median():.0f} s")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=False)
for ax, kitchen in zip(axes, ["Kitchen 1", "Kitchen 2", "Kitchen 3"]):
    sub = measurements[measurements.kitchen == kitchen]
    for bin_type, g in sub.groupby("bin"):
        ax.plot(g.timestamp, g.volume, lw=0.7, label=bin_type)
    ev = labels[labels.kitchen == kitchen]
    for t in ev.timestamp:
        ax.axvline(t, color="red", alpha=0.25, lw=0.8)
    ax.set_title(f"{kitchen} — nivel de llenado (%) · líneas rojas = vaciados etiquetados")
    ax.set_ylabel("volumen (%)")
    ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## 4 · Preprocesamiento: de señal de sensor a ventanas horarias

Los datos crudos son una señal irregular y ruidosa. Los convertimos en una tabla de **ventanas de
1 hora** por cocina y contenedor, que es la unidad de análisis del proyecto.

Para cada ventana calculamos:

| Feature | Qué es |
|---|---|
| `volume` | nivel de llenado medio en la hora (%) |
| `vol_max`, `vol_min` | máximo y mínimo en la hora |
| `n_readings` | cuántas lecturas hubo (proxy de calidad del dato) |
| `delta_1h` | cambio respecto a la hora anterior |
| `delta_3h` | cambio respecto a 3 horas antes |
| `mean_3h` | llenado medio de las últimas 3 horas |
| `hours_since_disposal` | horas desde el último vaciado **etiquetado** |
| `hour`, `dayofweek`, `is_weekend` | contexto temporal |

Y del `deployments.csv`: `service`, `area_m2`, `capacity_seats`, `bin_volume_m3`.

> **Por qué 1 hora:** a 5 minutos la señal es puro ruido de sensor; a 1 día solo quedan 249 filas.
> La hora deja ~3.300 ejemplos, que es lo mínimo razonable para afinar un encoder pequeño.

In [ ]:
# --- 4.1 Agregación a ventanas de 1 hora --------------------------------------
frames = []
for (kitchen, bin_type), g in measurements.groupby(["kitchen", "bin"]):
    r = g.set_index("timestamp")["volume"].resample("1h")
    f = pd.DataFrame({
        "volume": r.mean(),
        "vol_max": r.max(),
        "vol_min": r.min(),
        "n_readings": r.size(),
    }).reset_index()
    f["kitchen"], f["bin"] = kitchen, bin_type
    frames.append(f)

win = (pd.concat(frames, ignore_index=True)
       .sort_values(["kitchen", "bin", "timestamp"])
       .reset_index(drop=True))

print("Ventanas en la rejilla horaria completa:", len(win))
print("Ventanas con al menos una lectura      :", int((win.n_readings > 0).sum()))

In [ ]:
# --- 4.2 Features de historia (solo miran hacia atrás) ------------------------
g = win.groupby(["kitchen", "bin"])
win["delta_1h"] = win["volume"] - g["volume"].shift(1)
win["delta_3h"] = win["volume"] - g["volume"].shift(3)
win["mean_3h"] = g["volume"].transform(lambda s: s.rolling(3, min_periods=1).mean())

# lo que queremos predecir vive en t+1
win["next_volume"] = g["volume"].shift(-1)
win["next_n_readings"] = g["n_readings"].shift(-1)

# --- 4.3 Horas desde el último vaciado etiquetado -----------------------------
# merge_asof por cocina+contenedor: para cada ventana, el vaciado anterior más cercano.
win = win.sort_values("timestamp").reset_index(drop=True)
ev = (labels[["timestamp", "kitchen", "bin"]]
      .rename(columns={"timestamp": "last_disposal"})
      .sort_values("last_disposal")
      .reset_index(drop=True))

win = pd.merge_asof(
    win, ev,
    left_on="timestamp", right_on="last_disposal",
    by=["kitchen", "bin"], direction="backward",
)
win["hours_since_disposal"] = (win.timestamp - win.last_disposal).dt.total_seconds() / 3600

# --- 4.4 Contexto temporal ----------------------------------------------------
win["hour"] = win.timestamp.dt.hour
win["dayofweek"] = win.timestamp.dt.day_name()
win["is_weekend"] = win.timestamp.dt.dayofweek >= 5
win["date"] = win.timestamp.dt.date

win = win.sort_values(["kitchen", "bin", "timestamp"]).reset_index(drop=True)
print("Ventanas con hours_since_disposal conocido:",
      int(win.hours_since_disposal.notna().sum()), "de", len(win))
display(win.head())

## 5 · Definición de la tarea y del target

**Input:** el estado del contenedor hasta la hora `t`.
**Output:** el nivel de residuo que va a acumular **entre `t` y `t+1`**.

```
incremento = max(0, volume(t+1) − volume(t))
```

Recortamos en 0 porque una bajada de volumen no es "residuo negativo": es un vaciado o el personal
reacomodando la bolsa (el paper lo documenta explícitamente).

**Cortes de clase** (en puntos porcentuales de llenado):

| Clase | Incremento | Lectura operativa |
|---|---|---|
| `LOW` | ≤ 1 | el contenedor prácticamente no se movió |
| `MEDIUM` | 1 – 8 | acumulación normal de servicio |
| `HIGH` | > 8 | pico de generación; ojo, puede tocar vaciar pronto |

Son **cortes fijos e interpretables**, no terciles. Los terciles darían clases balanceadas
artificialmente pero sin significado operativo: al personal de cocina le importa el umbral real de
llenado, no el cuantil.

### 5.1 · Chequeo de leakage

`next_volume` y `increment_next_h` son el target. **Ninguno puede entrar en el texto de entrada.**
Todas las features listadas en la sección 4 se calculan con información disponible en `t` o antes.
Lo verificamos con un `assert` explícito.

In [ ]:
LOW_CUT, HIGH_CUT = 1.0, 8.0
LABEL_ORDER = ["LOW", "MEDIUM", "HIGH"]
label2id = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
id2label = {v: k for k, v in label2id.items()}

def to_label(inc):
    if inc <= LOW_CUT:
        return "LOW"
    if inc <= HIGH_CUT:
        return "MEDIUM"
    return "HIGH"

win["increment_next_h"] = (win["next_volume"] - win["volume"]).clip(lower=0)

# Solo ventanas con datos reales en t Y en t+1: si no hay lecturas no sabemos qué pasó.
data = win[(win.n_readings > 0) & (win.next_n_readings > 0)].copy().reset_index(drop=True)
data["label"] = data["increment_next_h"].apply(to_label)

print("Ventanas utilizables:", len(data))
dist = data.label.value_counts().reindex(LABEL_ORDER)
display(pd.DataFrame({"n": dist, "%": (dist / len(data) * 100).round(1)}))

print("\nIncremento (puntos porcentuales) por clase:")
display(data.groupby("label")["increment_next_h"].describe().reindex(LABEL_ORDER).round(2))

In [ ]:
# --- Chequeo de leakage: qué columnas puede ver el modelo ---------------------
TARGET_COLS = ["next_volume", "next_n_readings", "increment_next_h", "label"]
FEATURE_COLS = [
    "kitchen", "bin", "hour", "dayofweek", "is_weekend",
    "service", "area_m2", "capacity_seats", "bin_volume_m3",
    "volume", "delta_1h", "delta_3h", "mean_3h", "hours_since_disposal",
]

fuga = set(FEATURE_COLS) & set(TARGET_COLS)
assert not fuga, f"LEAKAGE: {fuga} está en features y en target"
print("OK: ninguna columna del target entra como feature.")
print("Features que verá el modelo:", FEATURE_COLS)

In [ ]:
# --- Metadatos de cada cocina desde deployments.csv ---------------------------
BIN_VOL_COL = {
    "Glass": "glass_volume",
    "Paper": "paper_volume",
    "Plastic": "plastic_volume",
    "Undifferentiated": "undifferentiated_volume",
}
dep_map = deployments.set_index("ID").to_dict("index")

def kitchen_meta(kitchen, bin_type):
    d = dep_map[int(kitchen.split()[-1])]
    return d["service"], d["area"], d["capacity"], d[BIN_VOL_COL[bin_type]]

meta = [kitchen_meta(k, b) for k, b in zip(data.kitchen, data["bin"])]
data[["service", "area_m2", "capacity_seats", "bin_volume_m3"]] = pd.DataFrame(meta, index=data.index)

display(data[["kitchen", "bin", "service", "area_m2", "capacity_seats", "bin_volume_m3"]]
        .drop_duplicates().reset_index(drop=True))

## 6 · Serialización: de la tabla al texto

Aquí está la decisión de diseño que mencionamos al principio. DistilBERT consume texto, así que
convertimos cada ventana en una descripción estructurada **en inglés** (el modelo base es
`distilbert-base-uncased`, entrenado en inglés — usar español lo penalizaría).

La plantilla es fija y determinista: mismas variables, mismo orden, misma redacción. Lo único que
cambia son los valores. **No se añade información que no estuviera en la tabla.**

**Cómo tratamos los valores faltantes.** Un 19% de las ventanas no tiene `delta_1h` o `delta_3h`
(son el arranque de cada serie o vienen después de un hueco en la señal), y un 9% no tiene
`hours_since_disposal` (los labels de vaciado están incompletos por diseño). En vez de imputar un 0
—que sería inventar "no hubo cambio"— **lo verbalizamos como ausencia**: el modelo lee explícitamente
que ese dato no está disponible. Cuesta unos tokens, pero no fabrica información y no obliga a tirar
642 filas.

In [ ]:
BIN_EN = {
    "Glass": "glass",
    "Paper": "paper",
    "Plastic": "plastic",
    "Undifferentiated": "undifferentiated (mostly food) waste",
}

def row_to_text(r):
    weekend = ", a weekend day" if r.is_weekend else ""
    partes = [
        f"Restaurant kitchen {r.kitchen.split()[-1]} serves {str(r.service).lower()}, "
        f"has {r.area_m2:.1f} square meters and seats {int(r.capacity_seats)} guests.",
        f"The monitored container holds {BIN_EN[r['bin']]} and its nominal volume is "
        f"{r.bin_volume_m3:.2f} cubic meters.",
        f"It is {r.dayofweek.lower()}{weekend} at {int(r.hour):02d}:00.",
        f"The container is currently {r.volume:.1f} percent full.",
    ]

    # Historia reciente: si el dato no existe, lo decimos en vez de imputar un 0.
    if pd.notna(r.delta_1h):
        partes.append(f"Over the last hour the fill level changed by {r.delta_1h:.1f} points.")
    else:
        partes.append("The change over the last hour is not available.")

    if pd.notna(r.delta_3h):
        partes.append(f"Over the last three hours it changed by {r.delta_3h:.1f} points.")
    else:
        partes.append("The change over the last three hours is not available.")

    partes.append(f"The average fill level in the last three hours was {r.mean_3h:.1f} percent.")

    if pd.notna(r.hours_since_disposal):
        partes.append(f"The container was last emptied {r.hours_since_disposal:.0f} hours ago.")
    else:
        partes.append("There is no record of when the container was last emptied.")

    return " ".join(partes)

data["text"] = data.apply(row_to_text, axis=1)

print("Valores faltantes verbalizados:")
for c in ["delta_1h", "delta_3h", "hours_since_disposal"]:
    n = int(data[c].isna().sum())
    print(f"  {c:22} {n:5} filas ({n / len(data) * 100:4.1f}%)")

print("\nLongitud del texto (palabras):")
print(data.text.str.split().str.len().describe().round(1).to_string())

print("\n--- EJEMPLOS (uno por clase, con historia completa) ---")
completos = data[data.delta_1h.notna() & data.delta_3h.notna()]
for lbl in LABEL_ORDER:
    ej = completos[completos.label == lbl].iloc[0]
    print(f"\n[{lbl}]  {ej.text}")

print("\n--- EJEMPLO con datos faltantes ---")
inc = data[data.delta_3h.isna() | data.hours_since_disposal.isna()]
print(inc.iloc[0].text)

## 7 · Split train / validation — **temporal, no aleatorio**

Este es un punto metodológico importante y es fácil equivocarse.

Un split aleatorio **fuga información** en series temporales: la ventana de las 14:00 y la de las
15:00 del mismo día comparten casi todo (`mean_3h`, `delta_3h` se solapan). Si una cae en train y
otra en validation, el modelo "recuerda" en vez de generalizar y la métrica sale inflada.

Usamos un **split temporal por día dentro de cada cocina**: el primer 80% de los días de cada cocina
va a train, el 20% final a validation. Así validamos sobre días que el modelo nunca vio, que es
justamente el escenario de uso real (predecir el futuro, no interpolar el pasado).

Guardamos el resultado en `data/processed/` para que el preprocesamiento sea auditable.

In [ ]:
train_idx = []
for kitchen, dd in data.groupby("kitchen"):
    dias = sorted(dd.date.unique())
    corte = dias[int(len(dias) * 0.8)]
    train_idx.append(dd.index[dd.date < corte])
    print(f"{kitchen}: {len(dias)} días · corte en {corte} · "
          f"train={int((dd.date < corte).sum())} val={int((dd.date >= corte).sum())}")

train_idx = np.concatenate(train_idx)
data["split"] = np.where(data.index.isin(train_idx), "train", "validation")

train_df = data[data.split == "train"].reset_index(drop=True)
val_df = data[data.split == "validation"].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Validation: {len(val_df)}")
print("\nDistribución de clases por split (%):")
display((pd.crosstab(data.split, data.label, normalize="index")[LABEL_ORDER] * 100).round(1))

In [ ]:
# Guardamos el dataset procesado (trazabilidad del preprocesamiento)
os.makedirs("data/processed", exist_ok=True)
COLS_OUT = ["kitchen", "bin", "timestamp", "date", "hour", "dayofweek", "is_weekend",
            "service", "area_m2", "capacity_seats", "bin_volume_m3",
            "volume", "vol_max", "vol_min", "n_readings",
            "delta_1h", "delta_3h", "mean_3h", "hours_since_disposal",
            "next_volume", "increment_next_h", "label", "split", "text"]
OUT_CSV = "data/processed/fikwaste_hourly_windows.csv"
data[COLS_OUT].to_csv(OUT_CSV, index=False)
print("Guardado:", OUT_CSV, data[COLS_OUT].shape)

## 8 · Modelo base y tokenizer

### ¿Por qué encoder-only?

De la Semana 3: el objetivo de preentrenamiento debe casar con la tarea.

- Los **encoder-only** (BERT, DistilBERT) se preentrenan con *masked language modeling*: atención
  **bidireccional**, ven toda la secuencia a la vez. Su fuerte es **clasificar y extraer**.
- Nuestra entrada es una descripción completa que hay que leer entera antes de asignarle una
  categoría. No generamos texto ni transformamos secuencias. → **encoder-only**.

### ¿Por qué DistilBERT y no BERT?

DistilBERT es un destilado de BERT-base: **~40% menos parámetros (66M vs 110M), ~60% más rápido**, y
retiene ~97% del desempeño en GLUE. Con ~2.700 ejemplos de entrenamiento y una T4 gratuita, el
cuello de botella no es la capacidad del modelo sino la cantidad de datos: gastar el doble de
cómputo en BERT-base no compraría nada.

### Inspección de tokenización del dominio

Consejo del Lab A: antes de casarse con un modelo base, hay que mirar **cómo tokeniza el vocabulario
del dominio**. Aquí es especialmente relevante porque nuestro "vocabulario" son **números**.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

ejemplo = train_df.loc[0, "text"]
tokens = tokenizer.tokenize(ejemplo)

print("Texto:", ejemplo[:160], "…")
print(f"\nTokens: {len(tokens)}")
print(tokens[:45])

print("\n--- Cómo trata el tokenizer nuestro vocabulario clave ---")
for term in ["undifferentiated", "container", "percent", "kitchen",
             "57.0", "8.0", "0.08", "13:00", "-2.4"]:
    print(f"  {term!r:20} -> {tokenizer.tokenize(term)}")

longs = [len(tokenizer.tokenize(t)) for t in train_df.text.sample(300, random_state=SEED)]
print(f"\nLongitud en tokens (muestra de 300): media {np.mean(longs):.0f}, máx {max(longs)}")
MAX_LEN = 192
print(f"MAX_LEN = {MAX_LEN} (cubre el máximo observado sin truncar)")

**Lectura de la tokenización (esto va al README):**

`undifferentiated` se parte en varias piezas de WordPiece, pero es un término que aparece en todas
las filas del mismo contenedor, así que el modelo lo aprende como patrón fijo. El problema real está
en los **números**: WordPiece fragmenta `57.0` y `-2.4` en varios tokens y **no preserva magnitud** —
para el modelo, `8.0` y `80.0` no están "cerca" en ningún sentido numérico.

Esta es la limitación estructural de meter datos de sensor por un tokenizer de texto, y es
exactamente la razón por la que en la sección 17 esperamos que un modelo de árboles —que sí trata los
números como números— sea competitivo o mejor.

## 9 · Baselines

La asignación exige comparación explícita. Usamos dos:

1. **Clase mayoritaria** — el piso absoluto. Sirve sobre todo para exponer el desbalance: con ~67%
   de `LOW`, un clasificador que siempre responde `LOW` saca ~0.67 de accuracy y parece decente.
   Su **F1-macro es ~0.27**, que revela que no sabe nada. Por eso nuestra métrica principal es
   F1-macro y no accuracy.

2. **DistilBERT sin fine-tuning (zero-shot)** — el baseline recomendado por la asignación. Usamos el
   objetivo original de MLM: le pedimos completar `The waste generation level is [MASK].` y
   comparamos las probabilidades de `low`, `medium` y `high`. Mide qué sabe el modelo base *antes* de
   que lo afinemos, así que el delta contra él es limpiamente atribuible a LoRA.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)

y_val = val_df["label"].tolist()

def evaluar(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# --- Baseline 1: clase mayoritaria -------------------------------------------
clase_mayoritaria = train_df["label"].value_counts().idxmax()
pred_mayoritaria = [clase_mayoritaria] * len(val_df)
baseline_majority = evaluar(y_val, pred_mayoritaria)

print(f"Clase mayoritaria en train: {clase_mayoritaria}")
display(pd.DataFrame([baseline_majority], index=[f"Majority ({clase_mayoritaria})"]).round(4))

In [ ]:
# --- Baseline 2: DistilBERT zero-shot (MLM, sin fine-tuning) ------------------
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained(BASE_MODEL).to(device).eval()
mask = tokenizer.mask_token

candidatos = {}
for palabra, clase in [("low", "LOW"), ("medium", "MEDIUM"), ("high", "HIGH")]:
    ids = tokenizer(palabra, add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        raise ValueError(f"'{palabra}' no es un token único: {ids}")
    candidatos[clase] = ids[0]
print("Token ids de las clases:", candidatos)

@torch.no_grad()
def zero_shot_batch(textos, batch_size=32):
    preds = []
    orden = [candidatos[c] for c in LABEL_ORDER]
    for i in range(0, len(textos), batch_size):
        prompts = [f"{t} The waste generation level is {mask}."
                   for t in textos[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LEN).to(device)
        logits = mlm(**enc).logits
        pos = (enc["input_ids"] == tokenizer.mask_token_id).float().argmax(dim=1)
        mask_logits = logits[torch.arange(logits.size(0)), pos]      # (B, vocab)
        elegido = mask_logits[:, orden].argmax(dim=1).cpu().numpy()
        preds.extend(LABEL_ORDER[i] for i in elegido)
    return preds

pred_zero = zero_shot_batch(val_df["text"].tolist())
baseline_zero = evaluar(y_val, pred_zero)

print("\nDistribución de lo que predice el zero-shot:",
      pd.Series(pred_zero).value_counts().to_dict())
display(pd.DataFrame([baseline_majority, baseline_zero],
                     index=[f"Majority ({clase_mayoritaria})", "DistilBERT zero-shot"]).round(4))

del mlm
torch.cuda.empty_cache() if device == "cuda" else None

## 10 · Dataset de Hugging Face y tokenización

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

def to_hf(df):
    d = Dataset.from_pandas(
        df[["text", "label"]].assign(labels=df["label"].map(label2id)).drop(columns=["label"]),
        preserve_index=False,
    )
    d = d.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    return d.remove_columns([c for c in d.column_names
                             if c not in ("input_ids", "attention_mask", "labels")])

train_tok, val_tok = to_hf(train_df), to_hf(val_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok)
print(val_tok)

## 11 · Configuración de LoRA

En vez de mover los 66M de parámetros de DistilBERT, **congelamos el modelo** y aprendemos dos
matrices de bajo rango por capa objetivo. Entrenamos **<1%** de los parámetros.

| Hiperparámetro | Valor | Por qué |
|---|---|---|
| `r` (rank) | `8` | Tenemos ~2.700 ejemplos y texto de plantilla fija (poca variación léxica). `r=8` da capacidad suficiente sin sobreajustar; `r=16+` con este volumen de datos memoriza. |
| `lora_alpha` | `16` | Regla estándar `alpha ≈ 2·r`: el factor de escala `alpha/r = 2` mantiene la magnitud de la actualización estable. |
| `lora_dropout` | `0.10` | Dataset pequeño y muy repetitivo → regularización explícita. |
| `target_modules` | `["q_lin", "v_lin"]` | Proyecciones de *query* y *value* de la atención en DistilBERT. Adaptar Q y V es el estándar del paper de LoRA: es donde se decide *a qué atiende* el modelo, que es lo que cambia entre dominios. |
| `modules_to_save` | `["pre_classifier", "classifier"]` | **Crítico.** La cabeza de clasificación de 3 clases se inicializa aleatoriamente al cargar el modelo. Si no la marcamos como entrenable, LoRA ajustaría el cuerpo contra una cabeza al azar y no aprende nada. |

> Los nombres de `target_modules` dependen del modelo: en DistilBERT son `q_lin`/`v_lin`, en
> BERT/RoBERTa `query`/`value`, en LLaMA/Qwen `q_proj`/`v_proj`.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label, label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.10,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 12 · Métrica y pesos de clase

**Métrica principal: F1-macro.** Promedia el F1 de las tres clases dándoles el mismo peso. Es la
correcta aquí porque:

- las clases están desbalanceadas (67 / 21 / 11);
- la clase **operativamente valiosa es `HIGH`** — es la que avisa de un pico de generación — y es la
  minoritaria. Con accuracy, acertar `HIGH` casi no mueve el número; con F1-macro pesa un tercio.

Reportamos también accuracy, precision-macro y recall-macro por contexto.

**Pesos de clase en la pérdida.** Con 67% de `LOW`, la salida degenerada "responder siempre `LOW`" es
un mínimo local muy cómodo. Ponderamos la *cross-entropy* con la frecuencia inversa de cada clase en
train para que equivocarse en `HIGH` cueste más.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels_ = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels_, preds),
        "precision_macro": precision_score(labels_, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels_, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels_, preds, average="macro", zero_division=0),
    }

conteo = train_df["label"].map(label2id).value_counts().sort_index()
pesos = torch.tensor((len(train_df) / (3 * conteo)).values, dtype=torch.float)
print("Conteo por clase en train:", {id2label[i]: int(c) for i, c in conteo.items()})
print("Pesos de clase          :", {id2label[i]: round(float(w), 3) for i, w in enumerate(pesos)})

## 13 · Entrenamiento con la Trainer API

`learning_rate=2e-4` es alto para fine-tuning completo pero es lo normal en LoRA: como solo se
mueven las matrices de bajo rango, hace falta un paso más grande para que el ajuste tenga efecto.

Cargamos el mejor checkpoint por `f1_macro` al final (`load_best_model_at_end`) para no quedarnos con
la última época si sobreajustó.

> El bloque usa `inspect.signature` para funcionar tanto en `transformers` 4.x (`evaluation_strategy`,
> `tokenizer=`) como en 5.x (`eval_strategy`, `processing_class=`).

In [ ]:
import inspect
from transformers import TrainingArguments, Trainer

class WeightedTrainer(Trainer):
    '''Trainer con cross-entropy ponderada por clase.'''
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_ = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels_, weight=pesos.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

OUTPUT_DIR = "./distilbert-lora-fikwaste"

ta_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=25,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)
ta_params = inspect.signature(TrainingArguments.__init__).parameters
clave = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"
ta_kwargs[clave] = "epoch"
ta_kwargs["save_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

tr_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
tr_params = inspect.signature(Trainer.__init__).parameters
tr_kwargs["processing_class" if "processing_class" in tr_params else "tokenizer"] = tokenizer

trainer = WeightedTrainer(**tr_kwargs)
print("Trainer listo. Épocas:", ta_kwargs["num_train_epochs"])

In [ ]:
train_result = trainer.train()
print("\nEntrenamiento terminado.")
print({k: round(v, 4) for k, v in train_result.metrics.items() if isinstance(v, float)})

## 14 · Evaluación final y comparación contra los baselines

Todo se evalúa sobre **exactamente el mismo** conjunto de validación (los últimos días de cada
cocina), que es lo que hace la comparación honesta.

In [ ]:
pred_out = trainer.predict(val_tok)
pred_lora = [id2label[int(i)] for i in np.argmax(pred_out.predictions, axis=-1)]
metrics_lora = evaluar(y_val, pred_lora)

comparacion = pd.DataFrame(
    [baseline_majority, baseline_zero, metrics_lora],
    index=[f"Baseline: clase mayoritaria ({clase_mayoritaria})",
           "Baseline: DistilBERT zero-shot",
           "DistilBERT + LoRA (afinado)"],
).round(4)
comparacion["Δ f1_macro vs zero-shot"] = (comparacion["f1_macro"]
                                          - baseline_zero["f1_macro"]).round(4)

print("=" * 78)
print("TABLA DE RESULTADOS — validación (últimos ~20% de días de cada cocina)")
print("=" * 78)
display(comparacion)

delta_zero = metrics_lora["f1_macro"] - baseline_zero["f1_macro"]
delta_may = metrics_lora["f1_macro"] - baseline_majority["f1_macro"]
print(f"\nΔ F1-macro vs zero-shot        : {delta_zero:+.4f}")
print(f"Δ F1-macro vs clase mayoritaria: {delta_may:+.4f}")

In [ ]:
# Historial de entrenamiento por época
hist = pd.DataFrame([h for h in trainer.state.log_history if "eval_f1_macro" in h])
if not hist.empty:
    display(hist[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(4))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(hist.epoch, hist.eval_f1_macro, marker="o", label="F1-macro (val)")
    ax.axhline(baseline_zero["f1_macro"], ls="--", c="gray", label="baseline zero-shot")
    ax.axhline(baseline_majority["f1_macro"], ls=":", c="red", label="baseline mayoritaria")
    ax.set_xlabel("época"); ax.set_ylabel("F1-macro"); ax.legend(); ax.grid(alpha=.3)
    ax.set_title("Evolución del F1-macro en validación")
    plt.tight_layout(); plt.show()

## 15 · F1 por clase y matriz de confusión

El F1-macro resume; la matriz de confusión dice **dónde** falla. La pregunta que importa: ¿el modelo
detecta los picos `HIGH`, o los está tragando dentro de `LOW`?

In [ ]:
print(classification_report(y_val, pred_lora, labels=LABEL_ORDER, zero_division=0, digits=3))

from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (nombre, preds) in zip(axes, [("DistilBERT zero-shot", pred_zero),
                                      ("DistilBERT + LoRA", pred_lora)]):
    cm = confusion_matrix(y_val, preds, labels=LABEL_ORDER)
    ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER).plot(
        ax=ax, values_format="d", cmap="Blues", colorbar=False)
    ax.set_title(f"{nombre}\nF1-macro = {f1_score(y_val, preds, average='macro', zero_division=0):.3f}")
plt.tight_layout()
plt.show()

## 16 · Ejemplos cualitativos (entrada → salida)

La asignación pide al menos 3. Mostramos aciertos y errores, porque los errores dicen más.

In [ ]:
ejemplos = val_df.copy()
ejemplos["prediccion"] = pred_lora
ejemplos["correcto"] = ejemplos.label == ejemplos.prediccion

def mostrar(df, titulo, n=3):
    print("\n" + "=" * 78)
    print(titulo)
    print("=" * 78)
    for _, r in df.head(n).iterrows():
        print(f"\nENTRADA : {r.text}")
        print(f"REAL    : {r.label}   ->   PREDICHO: {r.prediccion}"
              f"   {'✓' if r.correcto else '✗'}")
        print(f"(contexto: incremento real en la hora siguiente = "
              f"{r.increment_next_h:.2f} puntos)")

for lbl in LABEL_ORDER:
    sub = ejemplos[(ejemplos.label == lbl) & ejemplos.correcto]
    if not sub.empty:
        mostrar(sub, f"ACIERTO — clase real {lbl}", n=1)

mostrar(ejemplos[~ejemplos.correcto], "ERRORES DEL MODELO", n=3)

print("\n\nResumen de errores (real -> predicho):")
display(pd.crosstab(ejemplos.label, ejemplos.prediccion)
        .reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0))

## 17 · Reality check: ¿era el transformer la herramienta correcta?

Volvemos a la nota metodológica del principio. Entrenamos un `HistGradientBoostingClassifier` sobre
**las mismas features, sin pasar por texto**, con el **mismo split** y la misma métrica.

No es un baseline exigido por la asignación: es una comprobación de honestidad. Si el modelo de
árboles gana, eso *no invalida* el trabajo —el objetivo del módulo era dominar LoRA y la Trainer
API— pero sí es la conclusión correcta que debemos reportar en vez de esconder.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

NUM_FEATS = ["volume", "delta_1h", "delta_3h", "mean_3h", "hours_since_disposal",
             "hour", "area_m2", "capacity_seats", "bin_volume_m3", "n_readings"]

def matriz(df):
    X = df[NUM_FEATS].astype(float).copy()
    X["kitchen_id"] = df.kitchen.str[-1].astype(int)
    X["bin_id"] = df["bin"].map({b: i for i, b in enumerate(sorted(data["bin"].unique()))})
    X["dow_id"] = pd.to_datetime(df.timestamp).dt.dayofweek
    return X

gb = HistGradientBoostingClassifier(random_state=SEED, max_iter=200)
gb.fit(matriz(train_df), train_df.label.map(label2id))
pred_gb = [id2label[i] for i in gb.predict(matriz(val_df))]
metrics_gb = evaluar(y_val, pred_gb)

final = pd.DataFrame(
    [baseline_majority, baseline_zero, metrics_lora, metrics_gb],
    index=[f"Baseline: clase mayoritaria ({clase_mayoritaria})",
           "Baseline: DistilBERT zero-shot",
           "DistilBERT + LoRA (modelo de la entrega)",
           "[referencia] HistGradientBoosting tabular"],
).round(4)
display(final)

print(f"F1-macro  LoRA: {metrics_lora['f1_macro']:.4f}  |  "
      f"árboles: {metrics_gb['f1_macro']:.4f}  |  "
      f"diferencia: {metrics_lora['f1_macro'] - metrics_gb['f1_macro']:+.4f}")

## 18 · Guardar el adaptador LoRA

LoRA solo guarda las matrices de bajo rango y la cabeza de clasificación: unos pocos MB en vez del
modelo entero.

In [ ]:
ADAPTER_DIR = "./distilbert-lora-fikwaste-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

total_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
               for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"Adaptador guardado en {ADAPTER_DIR} ({total_mb:.1f} MB)")
print(os.listdir(ADAPTER_DIR))

## 19 · Resumen de resultados

La celda siguiente imprime el resumen que va copiado al README. **No escribas números a mano: sal
siempre de aquí.**

In [ ]:
print("=" * 78)
print("RESUMEN M1 — Fine-tuning LoRA sobre FIKWaste")
print("=" * 78)
print(f"Dataset          : FIKWaste (3 cocinas, Portugal, 2019) · CC BY 4.0")
print(f"Ventanas totales : {len(data)}  (train {len(train_df)} / validation {len(val_df)})")
print(f"Split            : temporal por día (primer 80% de días -> train)")
print(f"Modelo base      : {BASE_MODEL} (encoder-only, 66M parámetros)")
print(f"LoRA             : r=8, alpha=16, dropout=0.10, target=['q_lin','v_lin']")
print(f"Métrica principal: F1-macro")
print()
display(final)
print()
print(f"Δ F1-macro (LoRA − zero-shot)        : {delta_zero:+.4f}")
print(f"Δ F1-macro (LoRA − clase mayoritaria): {delta_may:+.4f}")

peor = min(LABEL_ORDER, key=lambda c: f1_score(
    [1 if y == c else 0 for y in y_val],
    [1 if p == c else 0 for p in pred_lora], zero_division=0))
print(f"Clase con peor F1                    : {peor}")

if delta_zero > 0:
    print("\nLectura: el fine-tuning con LoRA mejoró el F1-macro frente al modelo base sin afinar.")
else:
    print("\nLectura: el fine-tuning NO superó al baseline zero-shot. Revisar hiperparámetros, "
          "la representación textual y el tamaño del dataset antes de concluir.")

## 20 · Conclusiones

> **Completar después de ejecutar el notebook, con los números que imprimió la celda anterior.**
> No inventar resultados.

Responder:

1. **¿Superó el modelo afinado a los baselines?** ¿Cuánto, en F1-macro, contra cada uno?
2. **¿Qué clase quedó peor?** ¿Tiene sentido dado el desbalance (67/21/11) y el hecho de que `HIGH`
   es la minoritaria?
3. **¿Qué dice el reality check de la sección 17?** Si los árboles ganan, ¿por qué? (pista:
   WordPiece no preserva magnitud numérica — sección 8).
4. **¿Qué limitaciones del dataset explican el techo?** (~3.300 ventanas, 3 cocinas, 4 semanas,
   labels de vaciado incompletos, volumen % y no kg).
5. **¿Qué cambiarían en la siguiente iteración?**

---

### Limitaciones conocidas del dataset (obligatorio en la rúbrica, detalle en `docs/DATASET.md`)

- **Muestra minúscula y no representativa:** 3 cocinas de un mismo país, 4 semanas de 2019. No
  generaliza a otros tipos de restaurante, países ni temporadas.
- **Frecuencia de muestreo distinta entre cocinas** (1 min vs 5 min) → la Cocina 1 pesa más en el
  train de lo que le corresponde por días monitoreados.
- **Es volumen (%), no peso:** el sensor mide distancia. 10% de llenado de plástico y 10% de residuo
  orgánico no son la misma cantidad de desperdicio en kg.
- **El cero no es cero:** las bolsas vacías no se estiran del todo, así que un contenedor vacío puede
  marcar 20-30%. Esto sesga `volume` hacia arriba de forma distinta en cada contenedor.
- **Labels de vaciado incompletos por diseño** (solo eventos confirmados por ≥2 anotadores) →
  `hours_since_disposal` es ruidoso y a veces sobreestima.
- **`Undifferentiated` ≈ residuo alimentario, pero no exactamente:** es la fracción no reciclable, que
  incluye cosas que no son comida.

---

## 21 · Checklist de entrega M1

- [ ] El notebook corre de principio a fin en Colab gratuito (T4), sin errores.
- [ ] Modelo base y tokenizer cargados desde el Hub.
- [ ] Dataset del dominio cargado y preparado (secciones 2–7).
- [ ] Proceso de recolección y preprocesamiento incluido y documentado (secciones 2, 4, 6).
- [ ] Split train/validation explícito y justificado (temporal, sección 7).
- [ ] Chequeo de leakage con `assert` (sección 5.1).
- [ ] LoRA configurado y justificado: `r`, `alpha`, `target_modules` (sección 11).
- [ ] Trainer API usada, con métrica evaluada durante el entrenamiento.
- [ ] **Outputs del entrenamiento conservados** (no limpiar las celdas antes de subir).
- [ ] Baseline explícito ×2 y comparación sobre el mismo validation set (secciones 9 y 14).
- [ ] Métrica principal justificada: F1-macro (sección 12).
- [ ] Matriz de confusión y F1 por clase (sección 15).
- [ ] Al menos 3 ejemplos cualitativos entrada → salida (sección 16).
- [ ] Semilla fijada (`SEED = 42`) y celdas en orden.
- [ ] `README.md` abre con una frase entendible por alguien sin formación en IA.
- [ ] `docs/DATASET.md` con fuente, tamaño, idioma, licencia, tarea y **al menos un sesgo**.
- [ ] Conclusiones escritas con los números reales de la ejecución.

---

*SI4006 · Universidad EAFIT · Módulo 1 — Fine-tuning con LoRA · Dataset: FIKWaste (CC BY 4.0).*